# Multi-Format Extraction: Parquet, Excel, XML, YAML

document-graph supports multiple structured data formats beyond CSV and JSON.
This notebook demonstrates extraction from:

| Format | Provider | Use Case |
|--------|----------|----------|
| Parquet | `ParquetExtractProvider` | Data lakes (S3, Athena exports, Spark output) |
| Excel | `ExcelExtractProvider` | Business data, spreadsheets, manual inventories |
| XML | XML schema discovery | Legacy systems, SOAP APIs, configuration files |
| YAML | YAML schema discovery | Infrastructure-as-code, Kubernetes manifests |

All formats go through the same pipeline: Extract → Transform → Build.
The extraction stage normalizes everything into DataFrames regardless of source format.

**Prerequisites:**
- `pip install graphrag-toolkit-document-graph` (base install)
- `pip install openpyxl` (for Excel support)
- No Neptune or AWS credentials required
- Sample data in `data/` directory

## 1. Parquet Extraction

Parquet is the standard format for data lake outputs (Athena queries, Spark jobs, AWS Glue).
The `ParquetExtractProvider` reads columnar data efficiently and infers schema from column metadata.

In [ ]:
from graphrag_toolkit.document_graph.pipeline.extract.extract_provider_parquet import ParquetExtractProvider
from graphrag_toolkit.document_graph.pipeline.extract.extract_provider_config import ExtractProviderConfig
from graphrag_toolkit.document_graph.config import DocumentGraphConfig

config = ExtractProviderConfig(type='parquet', source='test_documents.parquet', document_id='parquet-demo')
provider = ParquetExtractProvider(config, DocumentGraphConfig())
result = provider.extract('data/test_documents.parquet')

print(f'Extracted: {result.dataframe.shape[0]} rows, {result.dataframe.shape[1]} columns')
print(f'Columns: {list(result.dataframe.columns)}')
print(f'Dtypes:\n{result.dataframe.dtypes}')
print(f'\nFirst 3 rows:')
result.dataframe.head(3)

### Parquet Schema Discovery

The schema discovery provider can infer an ETL schema directly from Parquet column metadata
(types, names, nullability) — no manual schema definition needed.

In [ ]:
from graphrag_toolkit.document_graph.schema.discovery.parquet_discovery_provider import ParquetSchemaDiscoveryProvider

discovery = ParquetSchemaDiscoveryProvider()
schema = discovery.discover('data/test_documents.parquet')

print('Discovered schema from Parquet metadata:')
print(f'  Fields: {len(schema.fields) if hasattr(schema, "fields") else "N/A"}')
print(f'  Schema: {schema}')

## 2. Excel Extraction

Excel files are common in business workflows — account lists, asset inventories, compliance records.
The `ExcelExtractProvider` reads `.xlsx` files and supports multi-sheet extraction.

In [ ]:
from graphrag_toolkit.document_graph.pipeline.extract.extract_provider_excel import ExcelExtractProvider

# Check if we have Excel sample data
import os
excel_files = [f for f in os.listdir('data') if f.endswith('.xlsx')]

if excel_files:
    excel_path = f'data/{excel_files[0]}'
    config = ExtractProviderConfig(type='excel', source=excel_files[0], document_id='excel-demo')
    provider = ExcelExtractProvider(config, DocumentGraphConfig())
    result = provider.extract(excel_path)
    
    print(f'Extracted from {excel_files[0]}:')
    print(f'  Rows: {result.dataframe.shape[0]}, Columns: {result.dataframe.shape[1]}')
    print(f'  Columns: {list(result.dataframe.columns)}')
    result.dataframe.head(3)
else:
    print('No .xlsx files in data/ — skipping Excel demo')
    print('To test: place any .xlsx file in the data/ directory')

## 3. XML Schema Discovery

XML is common in legacy systems, SOAP APIs, and configuration files.
The XML discovery provider parses the document structure to infer node types and relationships.

In [ ]:
from graphrag_toolkit.document_graph.schema.discovery.xml_discovery_provider import XMLSchemaDiscoveryProvider

discovery = XMLSchemaDiscoveryProvider()
schema = discovery.discover('data/test_documents.xml')

print('Discovered schema from XML structure:')
print(f'  Schema: {schema}')

## 4. YAML Schema Discovery

YAML is used in infrastructure-as-code (CloudFormation, Kubernetes), CI/CD configs, and documentation.
The YAML discovery provider extracts structure from the document hierarchy.

In [ ]:
from graphrag_toolkit.document_graph.schema.discovery.yaml_discovery_provider import YAMLSchemaDiscoveryProvider

discovery = YAMLSchemaDiscoveryProvider()
schema = discovery.discover('data/test_documents.yaml')

print('Discovered schema from YAML structure:')
print(f'  Schema: {schema}')

## 5. Pipeline: Parquet → Transform → Cypher

Complete pipeline from Parquet extraction through to Cypher generation — proving
that non-CSV formats work through the full document-graph pipeline.

In [ ]:
from graphrag_toolkit.document_graph import Node
from graphrag_toolkit.document_graph.graph_build.cypher_builder import node_to_cypher, batch_nodes_unwind
from graphrag_toolkit.document_graph.transform.transformer_provider_config import TransformerProviderConfig
from graphrag_toolkit.document_graph.transform.graph_transformers.row_to_node import RowToNodeTransformer

# Extract from Parquet
config = ExtractProviderConfig(type='parquet', source='test_documents.parquet', document_id='parquet-pipeline')
provider = ParquetExtractProvider(config, DocumentGraphConfig())
result = provider.extract('data/test_documents.parquet')
records = result.dataframe.to_dict('records')

# Transform: rows → nodes
transformer_config = TransformerProviderConfig(name='r2n', args={'type': 'Document'})
nodes = RowToNodeTransformer(transformer_config).transform(records)
print(f'Transformed {len(records)} Parquet rows → {len(nodes)} nodes')

# Build: generate Cypher
graph_nodes = [
    Node(id=n.get('id', str(i)), labels=['Document'], properties=n)
    for i, n in enumerate(nodes)
]
cypher, params = batch_nodes_unwind(graph_nodes, tenant_id='parquet_demo')

print(f'\nGenerated batch UNWIND query for {len(graph_nodes)} nodes:')
print(f'  Cypher: {cypher[:100]}...')
print(f'  Batch size: {len(params["batch"])}')
print(f'\n✅ Parquet → Transform → Cypher: complete')

## Summary

All structured formats go through the same pipeline:

```
Parquet/Excel/XML/YAML/CSV/JSON
    ↓ ExtractProvider (format-specific)
DataFrame (universal)
    ↓ Transformers (format-agnostic)
Node/Edge models
    ↓ Constructors
openCypher statements
    ↓ GraphStore
Neptune / Neo4j / FalkorDB
```

The extraction stage is the only format-specific code. Everything downstream is universal.